### Response Streaming
- the main problem when building chat applications using claude is the user experience problem
    - it takes 10-30 seconds to generate messages, causing the user to look at a loading spinner
- the solution is response streaming
    - lets users see text appear chunk by chunk as Claude generates
    - resulting in a more responsive feel

The Problem with Standard Responses:
- In a typical chat setup, your server sends a user message to Claude and waits for the complete response before anything is returned
- this causes a delay where users have no feedback or information on anything that is happening in the background

How Streaming Works
- when streaming is enabled, claude will immediately send a response saying that the request has been received, and is starting the text generation
- then a series of events is returned, each one containing a small chunk of the whole response
- these chunks of text can be forwarded to the application as soon as they arrive, allowing for the response to be built up word by word
- each one of those events are a single request to claude

Stream Events: When streaming is enabled, claude will send back multiple types of events
- MessageStart - A new message is being sent
- ContentBlockStart - Start of a new block containing text, tool use, or other content
- *ContentBlockDelta* - Chunks of the actual generated text (contains actual generated text)
- ContentBlockStop - The current content block has been completed
- MessageDelta - The current message is complete
- MessageStop - End of information about the current message


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
def chat(messages, system=None, temperature=1.0): 
    params = {
        "model": model,
        "max_tokens": 250,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

def add_user_message(messages, text): 
    user_message = {"role": "user", "content": text} 
    messages.append(user_message) 

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

In [5]:
messages = [] 

add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=250,
    messages=messages,
    stream=True # streaming is set to true in messages.create() for streaming to be enabled
)

for event in stream:
    print(event)
# contains everything including model used, roles, identifier id's, streaming events, and the message content

RawMessageStartEvent(message=Message(id='msg_011Ce1Jhuh3Nm6mckB1R7ere', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' FakeDB\n\nA lightweight in-memory database system that generates and stores randomly-populated tables with synthetic data for testing and', type='text_delta'), index=0,

In [10]:
# different way of creating a stream, that only returns the text from the response
# by automatically filters out everything except the actual text content

messages = []
add_user_message(messages, "Write a 2 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")
        
# this is different from the other ways of getting text back, since instead of waiting until all text has been returned, this method prints out text chunk by chunk as they are generated, creating a more seamless result and more closely mimics the generation of actual LLM platforms like claude, and gemini

stream.get_final_message() # complete message for storage or further processing
# this gets the final result, allowing the saving and storage of all messages creating a record for all individuals

# The CloudVault Archive Database

CloudVault Archive is a distributed NoSQL database designed to store and retrieve historical meteorological data from weather stations across the globe, with automatic compression and geo-spatial indexing to enable rapid queries about climate patterns spanning the past 150 years. The system uses a proprietary consensus algorithm called "TemporalSync" to maintain data consistency across 47 server clusters while allowing real-time writes from over 10,000 connected sensors worldwide.

ParsedMessage(id='msg_011Ce1MybG4QUTF7GSFudaSq', container=None, content=[ParsedTextBlock(citations=None, text='# The CloudVault Archive Database\n\nCloudVault Archive is a distributed NoSQL database designed to store and retrieve historical meteorological data from weather stations across the globe, with automatic compression and geo-spatial indexing to enable rapid queries about climate patterns spanning the past 150 years. The system uses a proprietary consensus algorithm called "TemporalSync" to maintain data consistency across 47 server clusters while allowing real-time writes from over 10,000 connected sensors worldwide.', type='text', parsed_output=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_t